**GOAL**


Write a prompt that will assist user in writing Python code, JSON config, or Regular Expressions focused on AWS-sepcific use cases

In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [13]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

Eval Dataset: Each object contain a 'task' that will merge into the prompt (create by Claude)

In [34]:
import json


def generate_dataset():
    system_prompt = """
Respond with raw JSON only.
Do not include markdown code fences (no ```json or ```).
Do not include any explanation or extra text before or after the JSON.
"""

    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluation the solution" 
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, system=system_prompt)
    return json.loads(text)

In [35]:
dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

[
  {
    "task": "Create a regular expression to validate AWS S3 bucket names according to AWS naming rules (lowercase letters, numbers, hyphens only, 3-63 characters, cannot start or end with hyphen)",
    "format": "regex",
    "solution_criteria": "Regex must match valid S3 bucket names and reject invalid ones (e.g., 'my-bucket-123' passes, 'My-Bucket' fails, '-bucket' fails, 'ab' fails)"
  },
  {
    "task": "Write a Python function that parses an AWS CloudWatch log entry and extracts the timestamp, log level, and message fields from a space-separated log line",
    "format": "python",
    "solution_criteria": "Function should correctly parse log entries like '2024-01-15T10:30:45Z ERROR Database connection timeout' and return a dictionary with timestamp, level, and message keys"
  },
  {
    "task": "Generate a JSON CloudFormation template snippet that defines an AWS Lambda function with basic execution role and environment variables for API_KEY and REGION",
    "format": "json",


In [36]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

___________________________________ 

Eval steps:
1. ask each task in dataset to Claude
2. grade each answer
3. modify prompt and repeat

**Grading**

Eval Criteria:
1. Format
2. Valid syntax
3. Task following

1 & 2 use code grader

3 use model grader

In [37]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria  you should use to evaluate  the solution
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    message = client.messages.create(
        model=model,
        max_tokens=4000,
        thinking={"type": "disabled"},
        messages=messages,
        output_config={
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "strengths": {"type": "array", "items": {"type": "string"}},
                        "weaknesses": {"type": "array", "items": {"type": "string"}},
                        "reasoning": {"type": "string"},
                        "score": {"type": "number"}
                    },
                    "required": ["strengths", "weaknesses", "reasoning", "score"]
                }
            }
        }
    )

    eval_text = message.content[0].text
    return json.loads(eval_text)

In [28]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [29]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    
    * Respond only with Python, JSON, or plain Regex
    * Do not add amy comments or commentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade['score']
    reasoning = model_grade['reasoning']
    
    syntax_score = grade_syntax(output, test_case)
    score = (syntax_score + model_score) / 2
    return {
        "output":output,
        "test_case":test_case,
        "score":score,
        "reasoning":reasoning
    }

In [22]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [42]:
with open("dataset.json", 'r') as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [43]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n```python\nimport re\n\ndef validate_s3_bucket_name(bucket_name):\n    \"\"\"\n    Validates AWS S3 bucket names according to AWS naming rules.\n    Rules:\n    - 3-63 characters long\n    - Can contain lowercase letters, numbers, and hyphens\n    - Cannot start or end with a hyphen\n    \"\"\"\n    pattern = r'^[a-z0-9]([a-z0-9-]{1,61}[a-z0-9])?$'\n    return bool(re.match(pattern, bucket_name))\n\n# Test cases\ntest_cases = [\n    (\"my-bucket\", True),\n    (\"my-bucket-123\", True),\n    (\"abc\", True),\n    (\"a\", False),  # Too short\n    (\"ab\", False),  # Too short\n    (\"my-bucket-\", False),  # Ends with hyphen\n    (\"-my-bucket\", False),  # Starts with hyphen\n    (\"my_bucket\", False),  # Contains underscore\n    (\"my.bucket\", False),  # Contains dot\n    (\"MyBucket\", False),  # Contains uppercase\n    (\"a\" * 64, False),  # Too long\n    (\"a\" * 63, True),  # Valid at max length\n    (\"1\" * 63, True),  # All numbers\n    (\"a-b-c-d-e-f

In [44]:
from statistics import mean
average_score = mean([result["score"] for result in results])
print(f'Average score: {average_score}')

Average score: 6.5
